# baseline v10.2 TEMPLATE — Qwen3-VL 8B/32B · H2000 · OCR 주입 · 앙상블

**팀 최고 LB 0.95978** (홀드아웃 0.9645) = 7B 제로샷 + 8B 학습(768) + 8B 학습(1024·4714) + 32B 학습(768·4714), **균등 평균** (09/22)

v10.2 변경: `USE_OCR_PROMPT` (질문 앞에 OCR 텍스트 주입, 제로샷 +1.2%p) · `RUN_ENSEMBLE` / `RUN_ZS_VALID` 스위치 → **밤샘 학습은 "모두 실행" 한 번이면 된다.**

**복사해서 쓰기**: 파일 → 드라이브에 사본 저장 → 이름을 `baseline_v10_2_본인이름` 으로.
0번 셀 `WHO`·`ZIP_PW` 채우기. **ZIP_PW 채운 노트북은 깃허브에 올리지 않는다.**

### 팀 공통 규칙 (앙상블하려면 꼭 지킬 것)
- `VALID_N = 2000`, `SEED = 42` **바꾸지 않는다** — 모두 같은 홀드아웃이어야 확률을 섞을 수 있다
- 실험이 끝나면 `valid_<TAG>.pt`(12번)와 `test_<TAG>_r0.pt`(13번)가 본인 폴더에 저장된다 → 성현에게 TAG 전달

| 셀 | 내용 | 비고 |
|---|---|---|
| 0~4 | 설정 · 설치 · 임포트 · 데이터 · 유형 | 세션 새로 열면 항상 |
| 5 | 프롬프트 | |
| 6 | 모델 로드 `load_model("7b")` | **모델 바꿀 땐 이 셀만** 다시 (메모리 정리 포함) |
| 7 | 추론 함수 | 이미지 로딩을 GPU 계산과 겹쳐서 빠르게 |
| 8 | 제로샷 (홀드아웃 + test) | 저장된 확률이 있으면 **다시 안 돌림** |
| 9 | **앙상블 → 제출** | GPU 모델 없이 파일만으로 돔. 0.939 재현 |
| 10~12 | Dataset · 학습 · 평가 | |
| 13~14 | test 추론 · 제출 체크 | |

홀드아웃은 **H2000** (train 셔플 앞 2000건, 오차 ±0.6%p). 기존 500건은 이 안에 포함된다.


## 0. 설정

In [ ]:
WHO        = ""            # 본인 이름 (영문). 저장 폴더가 된다
assert WHO, "0번 셀 WHO 에 본인 이름을 넣어주세요"
OFFLINE    = False
MODEL_SIZE = "q3_8b"         # 베이스. "7b" / "q3_8b"  (6번 셀 load_model 기본값)

LOSS_MODE       = "choice_ce"
SHUFFLE_CHOICES = True
GC_REENTRANT    = True       # gradient checkpointing 방식. 비전 LoRA 쓸 땐 False

IMAGE_SIZE = 768
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = IMAGE_SIZE * IMAGE_SIZE

VALID_N    = 2000            # 팀 공통. 바꾸지 말 것 (앙상블 기준 홀드아웃)
TRAIN_N    = 2000            # 최대 4714 (= 6714 - 홀드아웃 2000). 풀학습은 4714
EPOCHS     = 1
BATCH      = 1
GRAD_ACCUM = 8
LR         = 1e-4
LORA_R     = 16
EVAL_BATCH = 4
SEED       = 42
N_TTA      = 0               # 0=단독, 1=2-way, 3=4-way

ZIP_HINT = "ssafy-16-2"
ZIP_PW   = ""                # 압축 암호. 채운 채로 깃허브에 올리지 말 것
DATA_DIR = "/content/data"
OUT_DIR  = f"/content/drive/MyDrive/ssafy_ai/{WHO}"
SHARED   = "/content/drive/MyDrive/ssafy_ai/sunghyun"   # 팀 공유 확률 파일 (읽기만, 절대 덮어쓰지 말 것)

USE_RELABEL = False       # 은아 라벨 수정본을 학습 데이터에만 적용 (홀드아웃은 항상 원본)
RELABEL_CSV = "/content/drive/MyDrive/ssafy_ai/eunah/train_relabel.csv"   # id,old_answer,new_answer,reason

USE_OCR_PROMPT = False    # True: 질문 앞에 OCR 텍스트를 붙여서 학습·추론 (희원 캐시). TAG 끝에 _ocr
OCR_DIR        = "/content/drive/MyDrive/ssafy_ai/ocr"
OCR_MAX_CHARS  = 300

RUN_ZS_VALID = True       # 8번 홀드아웃 제로샷. 밤샘 학습에서 시간 아끼려면 False (저장 파일 있으면 어차피 몇 초)
RUN_ENSEMBLE = False      # 9번 앙상블. 학습 돌릴 땐 False, 앙상블만 할 땐 True
if OFFLINE:
    DATA_DIR, OUT_DIR = "./data", f"./runs/{WHO}"

_HF    = {"3b":    "Qwen/Qwen2.5-VL-3B-Instruct",
          "7b":    "Qwen/Qwen2.5-VL-7B-Instruct",
          "q3_8b": "Qwen/Qwen3-VL-8B-Instruct",
          "q3_32b": "Qwen/Qwen3-VL-32B-Instruct",   # 4bit 약 23GB. EVAL_BATCH=2 권장
          "q3_4b":  "Qwen/Qwen3-VL-4B-Instruct"}
_LOCAL = {"3b": "downloads/models/Qwen2.5-VL-3B-Instruct",
          "7b": "downloads/models/Qwen2.5-VL-7B-Instruct"}

def model_path(size):
    return _LOCAL[size] if OFFLINE else _HF[size]

def make_tag():
    return (f"{MODEL_SIZE}_{LOSS_MODE}_{'shuf' if SHUFFLE_CHOICES else 'noshuf'}"
            f"_img{IMAGE_SIZE}_r{LORA_R}_n{TRAIN_N}_e{EPOCHS}_H{VALID_N}"
            + ("_rl" if USE_RELABEL else "") + ("_ocr" if USE_OCR_PROMPT else ""))

print("OUT :", OUT_DIR, "| TAG:", make_tag())

## 1. 설치 (코랩)
설치 후 import 에러가 나면 **런타임 → 세션 다시 시작** 후 0번부터.

In [ ]:
import sys, subprocess
if not OFFLINE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.57.0", "accelerate", "peft>=0.13.2",
                    "bitsandbytes>=0.43.3", "pillow", "pandas"], check=True)
    print("설치 완료")

## 2. 임포트

In [ ]:
import os, sys, gc, math, random, glob, time, shutil, subprocess
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "GPU 런타임인지 확인"

USE_BF16   = torch.cuda.get_device_capability()[0] >= 8
AMP_DTYPE  = torch.bfloat16 if USE_BF16 else torch.float16
USE_SCALER = not USE_BF16

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 주의: 여기서 OUT_DIR makedirs 금지 (마운트 전에 가짜 로컬 폴더가 생긴다)
print("GPU :", torch.cuda.get_device_name(), "| AMP:", AMP_DTYPE)

## 3. 데이터 — 드라이브 마운트 + zip 해제 (약 40초)

In [ ]:
if not OFFLINE:
    from google.colab import drive
    MNT = "/content/drive"
    if not os.path.ismount(MNT):
        shutil.rmtree(MNT, ignore_errors=True)     # 마운트 안 된 가짜 로컬 폴더만 지움
        drive.mount(MNT)
    os.makedirs(OUT_DIR, exist_ok=True)

    if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
        hits = [p for p in glob.glob(f"{MNT}/MyDrive/**/*.zip", recursive=True)
                if ZIP_HINT in os.path.basename(p)]
        assert hits, "드라이브에서 zip 못 찾음 — ZIP_HINT 확인"
        assert ZIP_PW, "0번 셀 ZIP_PW 를 채워주세요"
        print("zip :", hits[0])
        t0 = time.time()
        os.makedirs(DATA_DIR, exist_ok=True)
        subprocess.run("apt-get -qq install -y p7zip-full", shell=True, check=False)
        r = subprocess.run(["7z", "x", "-y", f"-p{ZIP_PW}", f"-o{DATA_DIR}", hits[0]],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise SystemExit("압축 해제 실패 — ZIP_PW 확인\n" + (r.stdout or "")[-400:])
        print(f"해제 완료 {time.time()-t0:.0f}초")

# zip 안에 폴더가 한 겹 더 있으면 안쪽으로
if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    DATA_DIR = os.path.dirname(glob.glob(f"{DATA_DIR}/**/train.csv", recursive=True)[0])
print("DATA_DIR :", DATA_DIR, "| OUT_DIR :", OUT_DIR)

train_all = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
fix = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_DIR, str(p))
train_all["path"] = train_all["path"].map(fix)
test_df["path"]   = test_df["path"].map(fix)
assert os.path.exists(train_all["path"].iloc[0])

def inject_ocr(df, csv):
    o = pd.read_csv(csv)
    m = dict(zip(o["id"].map(os.path.basename), o["ocr_text"].fillna("").astype(str)))
    t = df["path"].map(os.path.basename).map(m).fillna("").str.strip()
    df = df.copy(); df["q_orig"] = df["question"]
    df["question"] = [f"[이미지에서 인식된 텍스트 참고]: {x[:OCR_MAX_CHARS]}\n\n{q}" if x else q
                      for x, q in zip(t, df["question"])]
    print(f"OCR 주입 {os.path.basename(csv)}: 글자 있음 {(t != '').mean():.1%}")
    return df

if USE_OCR_PROMPT:
    train_all = inject_ocr(train_all, f"{OCR_DIR}/ocr_train.csv")
    test_df   = inject_ocr(test_df,   f"{OCR_DIR}/ocr_test.csv")

# 셔플 순서는 SEED 로 고정 → 홀드아웃 500 ⊂ 1000 ⊂ 2000 (팀원 모두 같은 순서)
train_all = train_all.sample(frac=1, random_state=SEED).reset_index(drop=True)

def apply_relabel(df):
    r = pd.read_csv(RELABEL_CSV)
    m = dict(zip(r["id"].map(os.path.basename), r["new_answer"].astype(str).str.strip().str.lower()))
    k = df["id"].map(os.path.basename)
    hit = k.isin(m)
    df = df.copy(); df.loc[hit, "answer"] = k[hit].map(m)
    print(f"라벨 수정 적용: 학습 데이터 {hit.sum()}건")
    return df

def split():
    global valid_df, fit_df
    valid_df = train_all.iloc[:VALID_N].reset_index(drop=True)            # 홀드아웃은 항상 원본 라벨
    fit_df   = train_all.iloc[VALID_N:VALID_N + TRAIN_N].reset_index(drop=True)
    if USE_RELABEL:
        fit_df = apply_relabel(fit_df)
    print(f"홀드아웃 {len(valid_df)} / 학습 {len(fit_df)} / test {len(test_df)}")

split()      # VALID_N·TRAIN_N 바꾸면 split() 만 다시 호출

## 4. 질문 유형

In [ ]:
def qtype(q):
    q = str(q)
    if any(k in q for k in ["포함되지 않", "아닌 것", "없는 것", "없는 서비스", "제공하지 않",
                            "해당하지 않", "옳지 않", "올바르지 않", "잘못된", "틀린", "불가능한"]):
        return "부정형"
    if any(k in q for k in ["가격", "얼마", "요금", "금액", "원인가", "할인"]):
        return "가격·할인"
    if any(k in q for k in ["전화번호", "연락처", "번호는", "몇 층", "호실", "번지", "몇 번"]):
        return "번호·연락처"
    if any(k in q for k in ["시간", "언제", "기간", "날짜", "요일", "영업", "시부터", "발행일", "마감"]):
        return "시간·일정"
    if any(k in q for k in ["주소", "위치", "어디", "왼쪽", "오른쪽", "방면", "출구", "방향"]):
        return "위치·주소"
    if any(k in q for k in ["상호명", "브랜드", "가게 이름", "업체", "회사", "이름은", "명은", "제목"]):
        return "상호·이름"
    if any(k in q for k in ["메뉴", "음식", "제품", "모델", "상품"]):
        return "메뉴·상품"
    if any(k in q for k in ["적힌", "적혀", "쓰여", "쓰인", "문구", "글자",
                            "표기", "표시", "내용", "설명", "의미"]):
        return "문구·판독"
    if any(k in q for k in ["몇 개", "개수", "갯수"]):
        return "카운팅"
    if "색" in q:
        return "색상"
    return "기타"

QCOL = "q_orig" if "q_orig" in train_all.columns else "question"   # OCR 주입해도 유형은 원래 질문으로
print(train_all[QCOL].map(qtype).value_counts().to_string())

## 5. 프롬프트

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)
LETTERS = ["a", "b", "c", "d"]

def build_mc_prompt(question, a, b, c, d):
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(question, opts, img, answer=None):
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(question, *opts)},
        ]},
    ]
    if answer is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return msgs

## 6. 모델 로드

`load_model("q3_8b")` 처럼 부르면 **이전 모델을 메모리에서 내리고** 새로 올린다.
모델을 바꿀 때 이 셀만 다시 실행하면 되고, 7번 이후 함수는 다시 정의할 필요 없다.
학습된 어댑터로 추론만 할 때: `load_model("q3_8b", adapter=f"{OUT_DIR}/lora_<TAG>")`

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"

def _no_cache(m):
    # Qwen3-VL 은 text_config 안에도 use_cache 가 따로 있다. 켜져 있으면 학습 시 CheckpointError
    for cfg in [m.config, getattr(m.config, "text_config", None)]:
        if cfg is not None:
            cfg.use_cache = False

def load_model(size=None, adapter=None):
    global model, base_model, processor, tok, MODEL_SIZE, LETTER_IDS, LETTER_T, IM_END_ID
    # 학습 셀이 만든 변수들도 옛 모델을 붙잡고 있어서 같이 지운다 (안 지우면 32B 재로드 시 OOM)
    for n in ["model", "base_model", "optimizer", "scheduler", "scaler", "params", "out", "chosen",
              "raw_loss", "batch", "train_loader", "train_ds", "_check", "_ds", "enc", "logits"]:
        globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()
    if torch.cuda.memory_allocated() > 2e9:
        print(f"⚠ GPU에 {torch.cuda.memory_allocated()/1e9:.1f}GB 남아 있음 → 큰 모델이면 런타임 → 세션 다시 시작 후 0번부터")

    MODEL_SIZE = size or MODEL_SIZE
    path = model_path(MODEL_SIZE)
    processor = AutoProcessor.from_pretrained(path, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
                                              trust_remote_code=True)
    tok = processor.tokenizer

    _im = Image.open(train_all["path"].iloc[0]).convert("RGB")
    _n  = processor.image_processor(images=_im, return_tensors="pt")["pixel_values"].shape[0] // 4
    print(f"[{MODEL_SIZE}] {path} | 원본 {_im.size} -> 이미지 토큰 약 {_n}개")

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                             bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=AMP_DTYPE)
    base_model = AutoVLM.from_pretrained(path, quantization_config=bnb,
                                         device_map="auto", trust_remote_code=True)
    base_model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": GC_REENTRANT})
    _no_cache(base_model)

    if adapter:
        model = PeftModel.from_pretrained(base_model, adapter, is_trainable=False)
        print("어댑터 로드:", adapter)
    else:
        model = get_peft_model(base_model, LoraConfig(
            r=LORA_R, lora_alpha=LORA_R * 2, lora_dropout=0.05, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            task_type="CAUSAL_LM"))
        model.print_trainable_parameters()

    LETTER_IDS = [tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]
    LETTER_T   = torch.tensor(LETTER_IDS, device=device)
    IM_END_ID  = tok.convert_tokens_to_ids("<|im_end|>")
    assert [tok.decode([i]) for i in LETTER_IDS] == LETTERS, "정답 토큰 분해 확인 필요"
    print("LETTER_IDS:", LETTER_IDS, f"| GPU {torch.cuda.memory_allocated()/1e9:.1f}GB")

load_model(MODEL_SIZE)

## 7. 추론 함수 — 로짓 스코어링

다음 배치의 이미지 로딩·전처리를 **백그라운드 스레드에서 미리** 해 둔다 (GPU가 노는 시간 감소).
`save_path` 가 있으면 중간 저장하고, 이미 끝난 파일이면 **바로 불러와서 끝난다.**

In [ ]:
_POOL = ThreadPoolExecutor(max_workers=2)

def _prep(chunk, rotate):
    texts, images = [], []
    for _, row in chunk.iterrows():
        img  = Image.open(row["path"]).convert("RGB")
        opts = [row["a"], row["b"], row["c"], row["d"]]
        if rotate:
            opts = opts[rotate:] + opts[:rotate]
        texts.append(processor.apply_chat_template(
            build_messages(row["question"], opts, img), tokenize=False, add_generation_prompt=True))
        images.append(img)
    return processor(text=texts, images=images, padding=True, return_tensors="pt")

@torch.no_grad()
def predict_probs(df, batch_size=None, desc="predict", rotate=0, save_path=None, save_every=500):
    bs = batch_size or EVAL_BATCH
    model.eval(); tok.padding_side = "left"

    start, all_probs = 0, []
    if save_path and os.path.exists(save_path):
        done = torch.load(save_path, weights_only=False)
        all_probs, start = [done], done.shape[0]
        print(f"{os.path.basename(save_path)}: {start}/{len(df)}건 저장돼 있음")

    starts = list(range(start, len(df), bs))
    nxt = _POOL.submit(_prep, df.iloc[starts[0]:starts[0] + bs], rotate) if starts else None
    for k, s in enumerate(tqdm(starts, desc=desc)):
        enc = nxt.result()
        if k + 1 < len(starts):
            s2 = starts[k + 1]
            nxt = _POOL.submit(_prep, df.iloc[s2:s2 + bs], rotate)
        enc = enc.to(model.device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(**enc, use_cache=False).logits[:, -1, :]
        probs = logits[:, LETTER_IDS].float().softmax(-1).cpu()
        if rotate:                                          # 원래 보기 순서로 복원
            back = torch.zeros_like(probs)
            for j in range(4):
                back[:, (j + rotate) % 4] = probs[:, j]
            probs = back
        all_probs.append(probs)
        if save_path and (k + 1) % max(1, save_every // bs) == 0:
            torch.save(torch.cat(all_probs), save_path)

    out = torch.cat(all_probs)
    if save_path:
        torch.save(out, save_path)
    return out

def probs_to_letters(probs):
    return [LETTERS[i] for i in np.asarray(probs).argmax(-1).tolist()]

def acc_of(probs, df):
    gold = df["answer"].astype(str).str.strip().str.lower().values
    return float((np.array(probs_to_letters(probs)) == gold).mean())

def evaluate(df, name="valid", tta=0, save_path=None):
    probs = predict_probs(df, desc=f"eval[{name}]", save_path=save_path)
    for r in range(1, tta + 1):
        probs = probs + predict_probs(df, desc=f"eval[{name}]+r{r}", rotate=r)
    a = acc_of(probs, df)
    print(f"[{name}] accuracy = {a:.4f}  (n={len(df)}, tta={tta})")
    return a, probs

def report_by_qtype(df, probs, name=""):
    d = df.copy().reset_index(drop=True)
    d["pred"]    = probs_to_letters(probs)
    d["correct"] = d["pred"] == d["answer"].astype(str).str.strip().str.lower()
    d["qtype"]   = d[QCOL].map(qtype)
    print(f"--- {name} 유형별 ---")
    print(d.groupby("qtype")["correct"].agg(["mean", "count"]).round(3)
           .sort_values("mean").to_string())
    return d

def load_probs(p):
    x = torch.load(p, weights_only=False)
    if isinstance(x, dict):
        x = x.get("probs", x)
    return np.asarray(x)

## 8. 제로샷 (학습 없음)

홀드아웃·test 확률을 `OUT_DIR` 에 저장한다. **파일이 이미 있으면 불러오기만 한다.**
test 는 7B 약 36분 / 8B 약 18분 (추론 함수 개선 전 기준).

In [ ]:
_ocr     = "_ocr" if USE_OCR_PROMPT else ""
ZS_VALID = f"{OUT_DIR}/valid_zs_{MODEL_SIZE}_img{IMAGE_SIZE}_H{VALID_N}{_ocr}.pt"
_legacy  = f"{SHARED}/big2000_{MODEL_SIZE}.pt"           # 09/21 에 성현이 뽑아둔 H2000 확률 (OCR 없음)
if (VALID_N == 2000 and IMAGE_SIZE == 768 and not USE_OCR_PROMPT
        and os.path.exists(_legacy) and not os.path.exists(ZS_VALID)):
    shutil.copy(_legacy, ZS_VALID)

zs_acc = None
if RUN_ZS_VALID or os.path.exists(ZS_VALID):
    zs_acc, zs_probs = evaluate(valid_df, f"{MODEL_SIZE}-zeroshot{_ocr}", save_path=ZS_VALID)
    _ = report_by_qtype(valid_df, zs_probs, f"{MODEL_SIZE} zeroshot{_ocr}")
else:
    print("제로샷 건너뜀 (RUN_ZS_VALID=False)")

In [ ]:
RUN_ZS_TEST = False       # True 로 바꾸면 test 제로샷 확률 생성 (앙상블 재료)
if RUN_ZS_TEST:
    zs_test = predict_probs(test_df, desc=f"test-zs-{MODEL_SIZE}",
                            save_path=f"{OUT_DIR}/test_zeroshot_{MODEL_SIZE}_img{IMAGE_SIZE}{_ocr}.pt")
    print(pd.Series(probs_to_letters(zs_test)).value_counts().sort_index().to_dict())

## 9. 앙상블 → 제출 (**균등 평균이 기본**)

**GPU 모델이 없어도 된다** — 저장된 확률 파일(홀드아웃 H2000 + test)만 쓴다. 0~5번, 7번만 돌리고(6번 건너뜀) 여기로 와도 됨.

09/22 교훈: 홀드아웃 2000건으로 가중치를 탐색하면 과적합된다.
- 탐색 가중치: 홀드아웃 0.958 → 반분 검증 기대 0.9525 → **LB 0.95084**
- 균등 평균: 홀드아웃 0.9565 → LB 0.95174 → 멤버 교체 후 0.9645 → **LB 0.95978** (팀 최고)

**`0번 RUN_ENSEMBLE = True` 일 때만 돈다.**

새 멤버는 **균등 평균에 넣었을 때 홀드아웃이 오를 때만** 추가한다.
학습한 모델이 생기면 `ENS` 에 한 줄 추가 (`valid_<TAG>.pt`, `test_<TAG>_r0.pt`).
**제출 전 단톡에 공유** (팀 하루 20회 공유).

In [ ]:
JY  = "/content/drive/MyDrive/ssafy_ai/jeongyeon"
_P  = "choice_ce_shuf"
ENS = {   # 이름: (홀드아웃 H2000 확률, test 확률)   — LB 0.95978 구성
    "7b_zs":      (f"{SHARED}/big2000_7b.pt", f"{SHARED}/test_zeroshot_7b_img768.pt"),
    "8b_ft768":   (f"{SHARED}/valid_q3_8b_{_P}_img768_r16_n2000_e1_H2000.pt",
                   f"{SHARED}/test_q3_8b_{_P}_img768_r16_n2000_e1_H2000_r0.pt"),
    "8b_ft1024":  (f"{JY}/valid_q3_8b_{_P}_img1024_r16_n4714_e1_H2000.pt",
                   f"{JY}/test_q3_8b_{_P}_img1024_r16_n4714_e1_H2000_r0.pt"),
    "32b_ft4714": (f"{SHARED}/valid_q3_32b_{_P}_img768_r16_n4714_e1_H2000.pt",
                   f"{SHARED}/test_q3_32b_{_P}_img768_r16_n4714_e1_H2000_r0.pt"),
    # "내실험":   (f"{OUT_DIR}/valid_<TAG>.pt", f"{OUT_DIR}/test_<TAG>_r0.pt"),
    # "ocr":      (f"{SHARED}/valid_ocrmatch_H2000.pt", f"{SHARED}/test_ocrmatch.pt"),   # 현재 기여 0
}
if not RUN_ENSEMBLE:
    print("앙상블 건너뜀 (0번 RUN_ENSEMBLE=False)")
else:
    ens_valid = train_all.iloc[:2000].reset_index(drop=True)
    names = list(ENS)
    V = {k: load_probs(v) for k, (v, _) in ENS.items()}
    T = {k: load_probs(t) for k, (_, t) in ENS.items()}
    for k in names:
        assert V[k].shape == (2000, 4) and T[k].shape == (len(test_df), 4), (k, V[k].shape, T[k].shape)
        print(f"{k:12s} 홀드아웃 {acc_of(V[k], ens_valid):.4f}")

    # 균등 평균 + 하나씩 뺐을 때 (각 멤버의 기여도)
    W = {k: 1 / len(names) for k in names}
    s_all = acc_of(sum(W[k] * V[k] for k in names), ens_valid)
    print(f"\n균등 평균 ({len(names)}개): {s_all:.4f}")
    for drop in names:
        rest = [k for k in names if k != drop]
        s = acc_of(sum(V[k] for k in rest) / len(rest), ens_valid)
        print(f"  - {drop:12s} 빼면 {s:.4f}  (기여 {s_all - s:+.4f})")

    ens_test = sum(W[k] * T[k] for k in names)
    sub_name = "sub_ens_equal_" + "_".join(names) + f"_H2000_{s_all:.4f}.csv"
    submission = pd.DataFrame({"id": test_df["id"], "answer": probs_to_letters(ens_test)})
    submission.to_csv(f"/content/{sub_name}", index=False)
    submission.to_csv(f"{OUT_DIR}/{sub_name}", index=False)
    print("\n", sub_name, submission["answer"].value_counts().sort_index().to_dict())

In [ ]:
# (선택) 앙상블 홀드아웃 예측 저장 → 오답 분석용 (은아)
if RUN_ENSEMBLE:
    ens_v = sum(W[k] * V[k] for k in names)
    d = ens_valid.copy()
    d["qtype"]   = d[QCOL].map(qtype)
    d["pred"]    = probs_to_letters(ens_v)
    d["correct"] = d["pred"] == d["answer"].astype(str).str.strip().str.lower()
    d["conf"]    = ens_v.max(1).round(3)
    for k in names:
        d[f"pred_{k}"] = probs_to_letters(V[k])
    d["n_right"] = sum((d[f"pred_{k}"] == d["answer"]).astype(int) for k in names if k != "ocr")
    out = f"{OUT_DIR}/hold_pred_ens_equal_H2000_{s_all:.4f}.csv"
    d.drop(columns=["path"]).to_csv(out, index=False, encoding="utf-8-sig")
    print(out, "| 오답", int((~d["correct"]).sum()), "개")
    print(pd.crosstab(d["n_right"], d["correct"], margins=True))

## 10. Dataset / Collator (학습 시 보기 셔플 + choice_ce 라벨)

아래 검증에서 loss 토큰이 전부 a/b/c/d 이고, 셔플로 바뀐 샘플이 0/8 이 아니어야 한다.

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train=True, shuffle_choices=False):
        self.df, self.train, self.shuffle_choices, self.epoch = df.reset_index(drop=True), train, shuffle_choices, 0
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = Image.open(row["path"]).convert("RGB")
        opts = [str(row[l]) for l in LETTERS]
        gold = str(row["answer"]).strip().lower() if self.train else None
        if self.train and self.shuffle_choices:
            rng   = random.Random((SEED * 1000003) ^ (self.epoch * 7919) ^ i)
            order = list(range(4)); rng.shuffle(order)
            g_old = LETTERS.index(gold)
            opts  = [opts[k] for k in order]
            gold  = LETTERS[order.index(g_old)]
        text = processor.apply_chat_template(build_messages(row["question"], opts, img, answer=gold),
                                             tokenize=False, add_generation_prompt=False)
        return {"text": text, "image": img, "gold": gold}

@dataclass
class DataCollator:
    processor: Any
    def __call__(self, batch):
        self.processor.tokenizer.padding_side = "right"
        enc = self.processor(text=[b["text"] for b in batch], images=[b["image"] for b in batch],
                             padding=True, return_tensors="pt")
        labels, ans_pos = torch.full_like(enc["input_ids"], -100), []
        for i in range(len(batch)):
            ids  = enc["input_ids"][i]
            ends = (ids.eq(IM_END_ID) & enc["attention_mask"][i].bool()).nonzero().flatten()
            p    = ends[-1].item() - 1                   # 마지막 <|im_end|> 직전 = 정답 글자
            ans_pos.append(p); labels[i, p] = ids[p]
        enc["labels"]        = labels
        enc["ans_pos"]       = torch.tensor(ans_pos, dtype=torch.long)
        enc["choice_labels"] = torch.tensor([LETTERS.index(b["gold"]) for b in batch], dtype=torch.long)
        return enc

_ds    = VQAMCDataset(fit_df.head(8), train=True, shuffle_choices=SHUFFLE_CHOICES)
_check = DataCollator(processor)([_ds[i] for i in range(8)])
n_changed = 0
for i in range(8):
    p    = _check["ans_pos"][i].item()
    tokn = tok.decode([_check["input_ids"][i][p]])
    cl   = LETTERS[_check["choice_labels"][i].item()]
    orig = str(fit_df.iloc[i]["answer"]).strip().lower()
    n_changed += (orig != cl)
    print(f"{i}: 원본={orig} 셔플후={cl} loss토큰='{tokn}' {'OK' if tokn == cl else '<<< 이상'}")
print(f"셔플로 바뀐 샘플 {n_changed}/8")
assert all(tok.decode([_check['input_ids'][i][_check['ans_pos'][i].item()]]) in LETTERS for i in range(8))

## 11. 학습

A100 기준 8B 약 1.1 샘플/초 → TRAIN_N=2000 약 30분, 4714 약 70분.
200 스텝마다 어댑터 저장. **이전에 학습한 모델 위에 이어서 학습하지 않도록** 6번 `load_model()` 을 먼저 다시 부를 것.

In [ ]:
split()
TAG = make_tag()
print("TAG:", TAG)

train_ds = VQAMCDataset(fit_df, train=True, shuffle_choices=SHUFFLE_CHOICES)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          collate_fn=DataCollator(processor), num_workers=2)

model.train(); _no_cache(model)
params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR)
num_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, max(1, int(num_steps * 0.03)), num_steps)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
CKPT_DIR  = f"{OUT_DIR}/lora_{TAG}"
global_step, t0 = 0, time.time()

for epoch in range(EPOCHS):
    train_ds.epoch = epoch
    running, seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
    for step, batch in enumerate(pbar, start=1):
        batch         = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        ans_pos       = batch.pop("ans_pos")
        choice_labels = batch.pop("choice_labels")
        mask_labels   = batch.pop("labels")

        with torch.autocast("cuda", dtype=AMP_DTYPE):
            if LOSS_MODE == "mask":
                raw_loss = model(**batch, labels=mask_labels, use_cache=False).loss
            else:
                out    = model(**batch, use_cache=False)
                idx    = torch.arange(out.logits.size(0), device=device)
                chosen = out.logits[idx, ans_pos - 1, :].index_select(-1, LETTER_T)
                raw_loss = F.cross_entropy(chosen.float(), choice_labels)

        scaler.scale(raw_loss / GRAD_ACCUM).backward()
        running += raw_loss.item(); seen += 1

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad(set_to_none=True)
            global_step += 1
            pbar.set_postfix({"loss": f"{running/seen:.4f}"})
            running, seen = 0.0, 0
            if global_step % 200 == 0:
                model.save_pretrained(CKPT_DIR)

model.save_pretrained(CKPT_DIR); processor.save_pretrained(CKPT_DIR)
print(f"Saved: {CKPT_DIR}  ({(time.time()-t0)/60:.1f}분)")

## 12. 평가 — 제로샷보다 올랐나 (TTA 없이)

In [ ]:
ft_acc, ft_probs = evaluate(valid_df, TAG, save_path=f"{OUT_DIR}/valid_{TAG}.pt")
d_ft = report_by_qtype(valid_df, ft_probs, TAG)
d_ft.to_csv(f"{OUT_DIR}/hold_pred_{TAG}.csv", index=False)
if zs_acc is not None:
    print(f"\n제로샷 {zs_acc:.4f} -> 학습 {ft_acc:.4f}  ({ft_acc - zs_acc:+.4f})")
else:
    print(f"\n학습 {ft_acc:.4f}  (제로샷 비교 없음)")

# 7B 제로샷과 섞으면? (H2000 일 때만)
if VALID_N == 2000 and os.path.exists(f"{SHARED}/big2000_7b.pt"):
    b7 = load_probs(f"{SHARED}/big2000_7b.pt")
    for w in [0.3, 0.5, 0.7]:
        print(f"  + 7B제로샷 w={w}: {acc_of(w*b7 + (1-w)*np.asarray(ft_probs), valid_df):.4f}")

## 13. test 추론 + 제출 파일 (중간 저장, 끊기면 다시 실행하면 이어서)

In [ ]:
probs = predict_probs(test_df, desc="test-r0", save_path=f"{OUT_DIR}/test_{TAG}_r0.pt")
for r in range(1, N_TTA + 1):
    probs = probs + predict_probs(test_df, desc=f"test-r{r}", rotate=r,
                                  save_path=f"{OUT_DIR}/test_{TAG}_r{r}.pt")

sub_name = f"sub_{TAG}_tta{N_TTA+1}_hold{ft_acc:.4f}.csv"
submission = pd.DataFrame({"id": test_df["id"], "answer": probs_to_letters(probs)})
submission.to_csv(f"/content/{sub_name}", index=False)
submission.to_csv(f"{OUT_DIR}/{sub_name}", index=False)
print(sub_name, submission["answer"].value_counts().sort_index().to_dict())

## 14. 제출 전 체크 (9번·13번 어느 쪽이든)

In [ ]:
ss = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
print("행 수 일치 :", len(submission) == len(ss))
print("id 일치    :", (submission["id"].values == ss["id"].values).all())
print("결측       :", int(submission["answer"].isna().sum()))
ratio = submission["answer"].value_counts(normalize=True).sort_index()
print("분포       :", ratio.round(3).to_dict(), "| 최대 쏠림:", round(ratio.max(), 3))